# Checkpoint 4: One-Stage Anchor-Free Detector (YOLO26s)

Fine-tunes COCO-pretrained YOLO26s on VisDrone and evaluates it with the shared COCO protocol
(sequence-held-out validation split, every 3rd frame, ignored regions treated as "don't care").

Sections:
1. Setup
2. Configuration
3. Data
4. Training (resumes automatically from the last saved epoch)
5. Training curves
6. Validation metrics
7. Kaggle submission

## 1. Setup

Set `RUNTIME` to `"colab"` (checkpoints go to Google Drive) or `"local"` (checkpoints go to `runs/`).
On Colab, upload `kaggle.json` to the session first so the dataset can be downloaded.

In [ ]:
RUNTIME = "local"  # "colab" or "local"
REPO_URL = "https://github.com/rudkoLA/ai-hw-2-rudko-omelianchuk.git"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if RUNTIME == "colab":
    repo_dir = Path("/content/ai-hw-2-rudko-omelianchuk")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    uploaded_key = Path("/content/kaggle.json")
    if uploaded_key.exists():
        kaggle_key = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_key.parent.mkdir(parents=True, exist_ok=True)
        kaggle_key.write_bytes(uploaded_key.read_bytes())
        kaggle_key.chmod(0o600)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocotools", "kaggle"], check=True)

print(f"Working directory: {Path.cwd()}")

In [ ]:
import pandas as pd
from ultralytics import YOLO

from src.yolo import make_yolo_trainer
from src.setup_utils import ensure_dataset, resolve_storage, setup_environment
from src.submission import submit_to_kaggle
from src.ultralytics_utils import (
    evaluate_run,
    plot_training_curves,
    predict_submission,
    train_or_resume,
    write_yolo_split,
)

env = setup_environment(seed=42, mount_gdrive=False)
dataset_dir, coco_train, coco_val, class_names = ensure_dataset()
storage = resolve_storage(RUNTIME, "cp4_yolo")

## 2. Configuration

All tunable parameters. Give every experiment its own `run_name`: an existing run with the same name
is resumed (or skipped if finished) instead of being retrained.
Any other Ultralytics `train()` argument can be added to `TRAIN_EXTRA`.

In [ ]:
CFG = {
    "run_name": "yolo26s_640",
    "weights": "yolo26s.pt",      # COCO-pretrained
    "p2_head": False,              # True adds a stride-4 detection head (yolo26s-p2.yaml)
    "frame_stride": 3,
    # Schedule
    "imgsz": 640,
    "epochs": 30,
    "batch": 16,
    "optimizer": "MuSGD",
    "lr0": 0.01,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "warmup_epochs": 3,
    "warmup_bias_lr": 0.0,
    # Loss weights (for YOLO26 "dfl" weights the L1 box-distance term)
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    # Label assignment and box regression loss
    "tal_topk": 10,                # positives per object (one-to-many branch)
    "stal_min_side": None,         # GT sides below this (px) are enlarged for assignment; None = default, 0 = off
    "iou_type": "CIoU",            # "CIoU", "DIoU", "GIoU" or "IoU"
    # Inference
    "max_det": 500,
    "nms_free": False,             # True: one-to-one head without NMS; False: one-to-many head + NMS
    "nms_iou": 0.7,                # NMS IoU threshold (used when nms_free is False)
    # Misc
    "workers": 4,
    "seed": 42,
}

TRAIN_EXTRA = {}  # e.g. {"mosaic": 0.5, "close_mosaic": 5, "patience": 10}

## 3. Data

Writes Ultralytics image lists for every `frame_stride`-th frame of each sequence. The frame selection is
the same one used by `VisDroneDataset`, so all detectors are trained and evaluated on identical frames.

In [ ]:
data_yaml, frames = write_yolo_split(
    dataset_dir,
    {"train": coco_train, "val": coco_val},
    out_dir=storage / "lists",
    class_names=class_names,
    stride=CFG["frame_stride"],
)
del coco_train  # large; only the selected frames are needed from here on

## 4. Training

Checkpoints (`last.pt` every epoch, `best.pt`) and `results.csv` are written to `storage/<run_name>`.
Re-running this cell after an interruption resumes from `last.pt`; a finished run is not retrained.

In [ ]:
Trainer = make_yolo_trainer(
    tal_topk=CFG["tal_topk"],
    stal_min_side=CFG["stal_min_side"],
    iou_type=CFG["iou_type"],
)
run_dir = storage / CFG["run_name"]

train_or_resume(
    YOLO,
    "yolo26s-p2.yaml" if CFG["p2_head"] else "yolo26s.yaml",
    Trainer,
    run_dir,
    train_args={
        "data": str(data_yaml),
        "pretrained": CFG["weights"],
        **{k: CFG[k] for k in [
            "imgsz", "epochs", "batch", "optimizer", "lr0", "momentum", "weight_decay",
            "warmup_epochs", "warmup_bias_lr", "box", "cls", "dfl", "workers", "seed",
        ]},
        **TRAIN_EXTRA,
    },
    config={**CFG, "train_extra": TRAIN_EXTRA},
)

## 5. Training curves

Per-epoch validation mAP and losses logged by Ultralytics. These per-epoch metrics use Ultralytics' own
evaluator; the final numbers in Section 6 come from the shared COCO evaluation.

In [ ]:
history = plot_training_curves(run_dir)

## 6. Validation metrics

`best.pt` is evaluated on the validation frames with `evaluate_coco_metrics` (up to 500 detections per image;
with `nms_free=True` YOLO26 predicts with its one-to-one head and skips NMS). Results are saved to `storage/<run_name>/metrics.json`.

In [ ]:
PREDICT_KWARGS = {"nms": False} if CFG["nms_free"] else {"iou": CFG["nms_iou"]}

best_model = YOLO(str(run_dir / "weights" / "best.pt"))
metrics = evaluate_run(
    best_model,
    frames["val"],
    dataset_dir,
    imgsz=CFG["imgsz"],
    max_det=CFG["max_det"],
    batch=CFG["batch"],
    run_dir=run_dir,
    predict_kwargs=PREDICT_KWARGS,
)
pd.Series(metrics, name=CFG["run_name"]).round(4).to_frame()

## 7. Kaggle submission

Predicts all test frames with `best.pt` and writes `storage/<run_name>/submission.csv`
(boxes in original pixel coordinates, at most 500 detections per image, empty rows for frames without detections).

In [ ]:
submission_path = run_dir / "submission.csv"
report = predict_submission(
    best_model,
    dataset_dir,
    submission_path,
    imgsz=CFG["imgsz"],
    max_det=CFG["max_det"],
    batch=CFG["batch"],
    predict_kwargs=PREDICT_KWARGS,
)
report

Set `SUBMIT = True` to upload the file (requires `~/.kaggle/kaggle.json`).

In [ ]:
SUBMIT = False
SUBMISSION_MESSAGE = f"CP4 YOLO26s {CFG['run_name']}"

if SUBMIT:
    assert report["is_valid"], report["errors"]
    submit_to_kaggle(submission_path, SUBMISSION_MESSAGE)